In [7]:
import yfinance as yf
import pandas as pd

# Load master universe
master = pd.read_csv(r'D:\Data\Asset Allocation Project\master_securities.csv')

# Load client holdings input CSV
input_df = pd.read_csv(r'D:\Data\Asset Allocation Project\client_holdings.csv')

print(f"Loaded {len(input_df)} holdings across {input_df['Client'].nunique()} clients")

# Function to get live price
def get_price(ticker):
    try:
        return round(yf.Ticker(ticker).fast_info.last_price, 2)
    except:
        return None

# Pull prices for unique tickers only - faster
unique_tickers = input_df['Ticker'].unique()
print(f"Pulling prices for {len(unique_tickers)} unique tickers...")
price_map = {t: get_price(t) for t in unique_tickers}

# Map prices onto holdings
input_df['Price (£)'] = input_df['Ticker'].map(price_map)
input_df['Value (£)'] = input_df['Shares'] * input_df['Price (£)']

# Merge with master universe for Asset Class and Geography
input_df = input_df.merge(
    master[['Ticker', 'Long Name', 'Asset_Class', 'Geography']],
    on='Ticker',
    how='left'
)

# Calculate weight per client
input_df['Total Value (£)'] = input_df.groupby('Client')['Value (£)'].transform('sum')
input_df['Weight (%)'] = (input_df['Value (£)'] / input_df['Total Value (£)'] * 100).round(2)

# Print summary per client
for client in input_df['Client'].unique():
    df = input_df[input_df['Client'] == client]
    risk = df['Risk_Profile'].iloc[0]
    total = df['Total Value (£)'].iloc[0]
    
    print(f"\n{'='*45}")
    print(f"CLIENT: {client} | RISK: {risk} | VALUE: £{total:,.2f}")
    print(f"{'='*45}")
    
    asset_alloc = df.groupby('Asset_Class')['Weight (%)'].sum().sort_values(ascending=False)
    print("\nASSET CLASS ALLOCATION")
    for a, p in asset_alloc.items():
        print(f"  {a:<20} {p:.1f}%")
    
    geo_alloc = df.groupby('Geography')['Weight (%)'].sum().sort_values(ascending=False)
    print("\nGEOGRAPHIC ALLOCATION")
    for g, p in geo_alloc.items():
        print(f"  {g:<20} {p:.1f}%")

# Save full output
input_df.to_csv(r'D:\Data\Asset Allocation Project\portfolio_output.csv', index=False)
print("\n\nFull output saved to portfolio_output.csv")

# Clean final output for Power BI
powerbi_output = input_df[[
    'Client',
    'Risk_Profile', 
    'Ticker',
    'Long Name',
    'Shares',
    'Price (£)',
    'Value (£)',
    'Weight (%)',
    'Asset_Class',
    'Geography',
    'Total Value (£)'
]].copy()

# Round values
powerbi_output['Price (£)'] = powerbi_output['Price (£)'].round(2)
powerbi_output['Value (£)'] = powerbi_output['Value (£)'].round(2)
powerbi_output['Total Value (£)'] = powerbi_output['Total Value (£)'].round(2)

# Save
powerbi_output.to_csv(r'D:\Data\Asset Allocation Project\powerbi_feed.csv', index=False)

print(f"Power BI feed saved: {len(powerbi_output)} rows")
print(powerbi_output)

Loaded 13 holdings across 3 clients
Pulling prices for 8 unique tickers...

CLIENT: John Smith | RISK: Balanced | VALUE: £431,630.75

ASSET CLASS ALLOCATION
  UK Equity            76.6%
  Global Equity        17.9%
  Alternatives         5.1%
  Fixed Income         0.5%

GEOGRAPHIC ALLOCATION
  UK                   77.1%
  US                   13.3%
  Global               9.6%

CLIENT: Jane Doe | RISK: Cautious | VALUE: £27,742.00

ASSET CLASS ALLOCATION
  Fixed Income         34.4%
  Alternatives         33.0%
  UK Equity            32.6%

GEOGRAPHIC ALLOCATION
  UK                   67.0%
  Global               33.0%

CLIENT: Bob Jones | RISK: Growth | VALUE: £364,714.25

ASSET CLASS ALLOCATION
  Global Equity        45.7%
  UK Equity            45.3%
  Alternatives         9.0%

GEOGRAPHIC ALLOCATION
  UK                   45.3%
  US                   42.1%
  Global               12.6%


Full output saved to portfolio_output.csv
Power BI feed saved: 13 rows
        Client Risk_Profi